In [0]:
dbutils.widgets.text("run_date", "")

In [0]:
from pyspark.sql.functions import col, to_date
from datetime import datetime, timedelta
import boto3
import re

In [0]:
STAGE_PATH = "s3a://nyc-lakehouse/bronze/stage/"
BRONZE_PATH = "s3a://nyc-lakehouse/bronze/"
EXTRACT_PATH = "s3a://nyc-lakehouse/bronze/daily_extract/"
bucket = "nyc-lakehouse"

In [0]:

run_date_param = dbutils.widgets.get("run_date")

if run_date_param:
    run_date = datetime.strptime(run_date_param, "%Y-%m-%d").date()
else:
    # default
    run_date = (datetime.today() - timedelta(days=98)).date()

print(f"Running ingestion for date: {run_date}")


Running ingestion for date: 2025-11-01


In [0]:
stage_files = [f.path for f in dbutils.fs.ls(STAGE_PATH) if f.path.endswith(".parquet")]


In [0]:
year = run_date.year
month = f"{run_date.month:02d}"

source_path = (
    f"{STAGE_PATH}yellow_tripdata_{year}-{month}.parquet"
)

if source_path not in stage_files:
    dbutils.notebook.run('Extract_to_bronze', 60, arguments= {"date_code_param": f"{year}-{month}", "dest_folder": "stage/"})



In [0]:
raw_df = spark.read.parquet(source_path)

In [0]:
daily_df = (
    raw_df
    .filter(to_date(col("tpep_pickup_datetime")) == run_date)
)


In [0]:
temp_path = f"{EXTRACT_PATH}{run_date}"

# 1. Write to temp with 1 partition
daily_df.coalesce(1) \
    .write \
    .mode("overwrite") \
    .parquet(temp_path)


In [0]:

# Initialize client
ACCESS_KEY = dbutils.secrets.get(scope="my_app_creds", key="access_key_id")
SECRET_KEY = dbutils.secrets.get(scope="my_app_creds", key="secret_access_key")
REGION = 'us-east-1'

# Create a client with explicit credentials
s3 = boto3.client(
    's3',
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
    region_name=REGION
)

# 2. Find the generated part file
prefix = f"bronze/daily_extract/{run_date}/"
response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)

part_file_key = None
for obj in response.get("Contents", []):
    if re.search(r"part-.*\.parquet$", obj["Key"]):
        part_file_key = obj["Key"]
        break

if not part_file_key:
    raise Exception("Parquet part file not found!")

In [0]:
final_key = f"bronze/yellow_tripdata_{run_date}.parquet"

In [0]:
# 3. Move/rename to extract/data.parquet

s3.copy_object(
    Bucket=bucket,
    CopySource={"Bucket": bucket, "Key": part_file_key},
    Key=final_key
)

{'ResponseMetadata': {'RequestId': 'A4Q9Z9TV8B1XXF6H',
  'HostId': 'wjsSqV8janY8QOR2nbN1+yYu0hITIePr78XsLGLhZm00KZO2Q7ZbE2FKddKrl7zrt7ZKGqlllPj46gYwXBbofuKP17/VXC/iHJ2ePP/PioY=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'wjsSqV8janY8QOR2nbN1+yYu0hITIePr78XsLGLhZm00KZO2Q7ZbE2FKddKrl7zrt7ZKGqlllPj46gYwXBbofuKP17/VXC/iHJ2ePP/PioY=',
   'x-amz-request-id': 'A4Q9Z9TV8B1XXF6H',
   'date': 'Mon, 02 Feb 2026 03:10:59 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'content-type': 'application/xml',
   'content-length': '275',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'ServerSideEncryption': 'AES256',
 'CopyObjectResult': {'ETag': '"b8d0e367130941d7a42ca21f7805cfe1"',
  'LastModified': datetime.datetime(2026, 2, 2, 3, 10, 59, tzinfo=tzlocal()),
  'ChecksumCRC64NVME': 'CZgmj1SIl8o='}}

In [0]:
# 4. Clean up temp folder
for obj in response.get("Contents", []):
    s3.delete_object(Bucket=bucket, Key=obj["Key"])

print(f"✅ Single parquet file written to s3://{bucket}/{final_key}")

✅ Single parquet file written to s3://nyc-lakehouse/bronze/yellow_tripdata_2025-11-01.parquet
